# 04 — Reporting Views: Project Profitability & Utilisation Analytics
Creates SQL views over Gold tables for Power BI and ad-hoc analysis.

| View | Purpose |
|---|---|
| vw_executive_summary | Single-row KPI snapshot |
| vw_low_margin_projects | Engagements with margin < 5% |
| vw_underutilised_consultants | Consultants with utilisation < 60% |
| vw_client_revenue_concentration | Client revenue share ranked |

In [ ]:
# Views are created over the gold tables in the attached lakehouse (dbo schema).


In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW vw_executive_summary AS
SELECT
    SUM(engagements)                                                     AS total_engagements,
    ROUND(AVG(avg_margin_pct), 2)                                        AS avg_margin_pct,
    SUM(at_risk_engagements)                                             AS at_risk_engagements,
    SUM(consultant_count)                                                AS total_consultants,
    ROUND(SUM(total_billed_gbp), 0)                                      AS total_revenue_gbp
FROM gold_portfolio_scorecards
""")
print("vw_executive_summary created")

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW vw_low_margin_projects AS
SELECT
    engagement_id,
    client_id,
    practice,
    status,
    budget_gbp,
    actual_spend_gbp,
    margin_pct,
    margin_band,
    overrun_pct
FROM gold_project_profitability
WHERE margin_pct < 5
ORDER BY margin_pct ASC
""")
print("vw_low_margin_projects created")

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW vw_underutilised_consultants AS
SELECT
    consultant_id,
    grade,
    practice,
    region,
    grade_band,
    utilisation_rate_pct,
    billable_hours,
    total_billed_gbp
FROM gold_consultant_utilisation
WHERE utilisation_rate_pct < 60
ORDER BY utilisation_rate_pct ASC
""")
print("vw_underutilised_consultants created")

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW vw_client_revenue_concentration AS
SELECT
    client_id,
    client_name,
    tier,
    industry,
    total_revenue_gbp,
    concentration_pct,
    avg_margin_pct,
    nps_score,
    nps_band
FROM gold_client_revenue
ORDER BY total_revenue_gbp DESC
""")
print("vw_client_revenue_concentration created")

In [ ]:
print("\n=== Reporting Views Complete ===")
for v in ["vw_executive_summary", "vw_low_margin_projects", "vw_underutilised_consultants", "vw_client_revenue_concentration"]:
    spark.sql(f"SELECT * FROM {v}").show(3, truncate=False)